Why Coinapi?
- Delisted symbols
- Long history (back to the origins)
- Cleaned data
- Pricing quite fair and usage-based (Coinmarketcap costs $700/month for the same)

In [1]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
from datetime import datetime, date
import os
from matplotlib import pyplot as plt
import coinapi_fetcher
import time
from dotenv import load_dotenv

load_dotenv()

exchange = os.getenv('COINAPI_EXCHANGE')
base_currency = os.getenv('COINAPI_BASE_CURRENCY')

output_folder = f'../{exchange.lower()}_data/'
print(f'Store files in {output_folder}')

Store files in ../binance_data/


In [2]:
# btc = coinapi_fetcher.get_history('COINBASE_SPOT_BTC_USD', datetime(2010, 1, 1))
# btc['close'].plot(figsize=(15, 6))
# plt.show()
# btc.to_csv(os.path.join(output_folder, 'COINBASE_SPOT_BTC_USD.csv'), index=True)

In [3]:
# Binance doesn't have base currency USD; use BTC and re-caculate (BINANCE has a market share
# of approx 40%)
# Problem: There is no BTC-BTC, so we don't get the BTC volume. Use Coinbase instead, as they
# directly quote USD.
exchange_id = exchange
active = coinapi_fetcher.get_active_symbols(exchange_id, base_currency=base_currency)
historical = coinapi_fetcher.get_historical_symbols(exchange_id, base_currency=base_currency)

print(f'Got {len(active)} active and {len(historical)} historical symbols')
print(f'Active symbols: {active}.to_list()')
print(f'Historical symbols: {historical}.to_list()')

Get active symbols for BINANCE


Get historical symbols for BINANCE, adding to 0 existing


Out of 1000 historical, 183 are USDT-based
Get historical symbols for BINANCE, adding to 183 existing


Out of 1000 historical, 187 are USDT-based
Get historical symbols for BINANCE, adding to 370 existing


Out of 1000 historical, 176 are USDT-based
Get historical symbols for BINANCE, adding to 546 existing


Out of 531 historical, 104 are USDT-based
Got 438 active and 650 historical symbols
Active symbols: ['BINANCE_SPOT_FORTH_USDT', 'BINANCE_SPOT_ARB_USDT', 'BINANCE_SPOT_ACH_USDT', 'BINANCE_SPOT_ESP_USDT', 'BINANCE_SPOT_MINA_USDT', 'BINANCE_SPOT_CAKE_USDT', 'BINANCE_SPOT_GLMR_USDT', 'BINANCE_SPOT_XLM_USDT', 'BINANCE_SPOT_BEL_USDT', 'BINANCE_SPOT_ASTER_USDT', 'BINANCE_SPOT_SAGA_USDT', 'BINANCE_SPOT_COW_USDT', 'BINANCE_SPOT_MUBARAK_USDT', 'BINANCE_SPOT_BAR_USDT', 'BINANCE_SPOT_1000SATS_USDT', 'BINANCE_SPOT_GRT_USDT', 'BINANCE_SPOT_HOME_USDT', 'BINANCE_SPOT_RUNE_USDT', 'BINANCE_SPOT_ETH_USDT', 'BINANCE_SPOT_TLM_USDT', 'BINANCE_SPOT_ZEN_USDT', 'BINANCE_SPOT_TKO_USDT', 'BINANCE_SPOT_PIVX_USDT', 'BINANCE_SPOT_ASTR_USDT', 'BINANCE_SPOT_FRAX_USDT', 'BINANCE_SPOT_NEAR_USDT', 'BINANCE_SPOT_FF_USDT', 'BINANCE_SPOT_CITY_USDT', 'BINANCE_SPOT_FLUX_USDT', 'BINANCE_SPOT_ALT_USDT', 'BINANCE_SPOT_VTHO_USDT', 'BINANCE_SPOT_AVNT_USDT', 'BINANCE_SPOT_BOME_USDT', 'BINANCE_SPOT_ORDI_USDT', 'BINANCE_SPOT_LINEA_U

In [4]:
def write_file(data, file_path):
    data.to_csv(file_path, index=True)
    print(f'Wrote file {file_path} with {len(data)} rows')

def get_existing_content(file_path):
    '''
    Returns the date of the last row in the file, if it exists, and its whole content (which
    is needed to append new data to later).
    '''
    if os.path.exists(file_path):
        print(f'File {file_path} exists, get last row')
        content = pd.read_csv(file_path)
        # When the file is empty, there's no 'date' column that we can use as index; return an
        # empty df so that all data is fetched from the crypto's start of existence.
        if ('date' not in content.columns):
            print('File is missing date column, return empty DataFrame')
            return (pd.DataFrame(), None)
        # Remove the last row; it may contain intraday data; make it 2 to be sure.
        content = content.iloc[:-2]
        content['date'] = pd.to_datetime(content['date'])
        content.set_index('date', inplace=True)
        print(f'Existing file has {len(content)} rows')
        if(len(content)):
            last_date = content.index[-1].date()
            print(f'Last date in existing file is {last_date}')
            return (content, last_date)
        return (content, None)
    else:
        return (pd.DataFrame(), None)

first_date = date(2010, 1, 1)

current_index = 0
# Historical contains *all* cryptos, delisted as well as active. Delisted ones just won't return
# any new data.
for symbol_id in historical:
    current_index += 1
    print('------')
    print(f'Get {current_index}/{len(historical)}')
    file_path = os.path.join(output_folder, 'historical', f'{symbol_id}.csv')
    (existing_content, start_date) = get_existing_content(file_path)
    if (start_date):
        print(f'File {symbol_id} exists, 2nd latest row\'s date is {start_date}')
    else:
        start_date = first_date
        print(f'File {symbol_id} does not exist')
    end_date = date.today()
    if (start_date >= end_date):
        print(f'Skip {symbol_id}, start is on or after end')
        continue
    print(f'Get {symbol_id} from {start_date} to {end_date}')
    new_content = coinapi_fetcher.get_history(symbol_id, start_date, end_date)
    # Make sure that content was returned before we concat an empty DF (which would fail)
    if (not new_content.empty):
        data = pd.concat([df for df in [existing_content, new_content] if not df.empty])
        write_file(data, file_path)
    time.sleep(0.5)
print(f'Out of {len(historical)} cryptos, {current_index} were fetched')

------
Get 1/650
File ../binance_data/historical/BINANCE_SPOT_BCC_USDT.csv exists, get last row
Existing file has 2045 rows
Last date in existing file is 2023-08-28
File BINANCE_SPOT_BCC_USDT exists, 2nd latest row's date is 2023-08-28
Get BINANCE_SPOT_BCC_USDT from 2023-08-28 to 2026-03-02
Fetching BINANCE_SPOT_BCC_USDT from 2023-08-28 to 2026-03-02 …


Fetched 2 bars
Wrote file ../binance_data/historical/BINANCE_SPOT_BCC_USDT.csv with 2047 rows


------
Get 2/650
File ../binance_data/historical/BINANCE_SPOT_BNB_USDT.csv exists, get last row
Existing file has 3032 rows
Last date in existing file is 2026-02-25
File BINANCE_SPOT_BNB_USDT exists, 2nd latest row's date is 2026-02-25
Get BINANCE_SPOT_BNB_USDT from 2026-02-25 to 2026-03-02
Fetching BINANCE_SPOT_BNB_USDT from 2026-02-25 to 2026-03-02 …


Fetched 5 bars
Wrote file ../binance_data/historical/BINANCE_SPOT_BNB_USDT.csv with 3037 rows


------
Get 3/650
File ../binance_data/historical/BINANCE_SPOT_BTC_USDT.csv exists, get last row
Existing file has 3113 rows
Last date in existing file is 2026-02-25
File BINANCE_SPOT_BTC_USDT exists, 2nd latest row's date is 2026-02-25
Get BINANCE_SPOT_BTC_USDT from 2026-02-25 to 2026-03-02
Fetching BINANCE_SPOT_BTC_USDT from 2026-02-25 to 2026-03-02 …


...

------
Get 649/650
File ../binance_data/historical/BINANCE_SPOT_U_USDT.csv exists, get last row
Existing file has 43 rows
Last date in existing file is 2026-02-25
File BINANCE_SPOT_U_USDT exists, 2nd latest row's date is 2026-02-25
Get BINANCE_SPOT_U_USDT from 2026-02-25 to 2026-03-02
Fetching BINANCE_SPOT_U_USDT from 2026-02-25 to 2026-03-02 …


Fetched 5 bars
Wrote file ../binance_data/historical/BINANCE_SPOT_U_USDT.csv with 48 rows


------
Get 650/650
File ../binance_data/historical/BINANCE_SPOT_ZAMA_USDT.csv exists, get last row
Existing file has 23 rows
Last date in existing file is 2026-02-25
File BINANCE_SPOT_ZAMA_USDT exists, 2nd latest row's date is 2026-02-25
Get BINANCE_SPOT_ZAMA_USDT from 2026-02-25 to 2026-03-02
Fetching BINANCE_SPOT_ZAMA_USDT from 2026-02-25 to 2026-03-02 …


Fetched 5 bars
Wrote file ../binance_data/historical/BINANCE_SPOT_ZAMA_USDT.csv with 28 rows


Out of 650 cryptos, 650 were fetched


In [5]:
# Turns out: /history returns active *and* delisted data
for symbol_id in active:
    if (symbol_id not in historical):
        print(f'!!!!! Active symbol {symbol_id} not in historical list')
#     file_path = f'{output_folder}/active/{symbol_id}.csv'
#     start_date = get_latest_entry_date(file_path) or first_date
#     end_date = date.today()
#     if (start_date >= end_date):
#         print(f'Skip {symbol_id}, start is on or after end')
#         continue
#     print(f'Get {symbol_id} from {start_date}')
#     data = coinapi_fetcher.get_history(symbol_id, start_date)
#     write_file(data, file_path)
#     time.sleep(1)

# print('Done')